In [0]:
import os
import re
import json
import pandas
import numpy
import datetime
import pprint
import chardet

def getExcelRegionInIntegers(region):
    '''This function takes the region info from the config and returns a list of start / end row / column numbers.
        It alphanumeric excel based cell index to list[row, col] index ==> ABZ987 = [987, 104]'''

    #Cast to upper case
    region = region.upper()

    #Split based on colon":".  Assumed that the format is
    #[start-row-col:end-row-col]
    rowColList = region.split(":")

    #Removing duplicatesahrefs_ctr.xlsx
    rowColList = list(dict.fromkeys(rowColList))        
        

    rowColumnList = list()

    for val in rowColList:
        #Setting/Reseting the row-column initializations for each of regions in the list passed
        row = ''
        colIndex = -1

        #Parsing each region in reverse
        for ch in val[::-1]:
            if ch.isalpha():
                #Logic to convert columns with Aplhabets to integers.  Assumed
                #that columns will be indicated as alphabets and rows as
                #numbers.
                #A --> 1 | Z --> 26 | AA --> 27 ==> (26 * 1) + 1 = 27 | AZ -->
                #52 ==> (26 * 1) + 26 = 52 | AAA --> 53 ==> (26 * 1) + (26 * 1)
                #+ 1 = 53
                #AAZ --> 78 ==> (26 * 1) + (26 * 1) + 26 | ABA --> 79 ==> (26 *
                #1) + (26 * 2) + 1 = 79 | ABZ --> 104 ==> (26 * 1) + (26 * 2) +
                #26 = 104
                #Format: ((26 * {1 based numerical index of alphabet}) * n) +
                #{1 based numerical index of alphabet}
                #Parsing in the reverse direction, only the right most
                #alphabet's index will be conidered.  The subsequent alphabets
                #indexs will be multiplied with 26 (alphabet count).
                number = ord(ch) - 65
                if colIndex > -1 :
                    colIndex += ((number + 1) * 26)
                else:
                    colIndex = number

            elif ch.isdigit():
                row = ch + row

        row = int(row)
        row = row - 1
        if row < 0 : row = 0
        rowColumnList.append({"colIndex" : colIndex, "rowIndex" : row})

    #When reading values from one cell, the start and end indexes would be the same
    if len(rowColumnList) == 1:
        endRowColumnIndex = {"colIndex" : rowColumnList[0]["colIndex"] + 1, "rowIndex" : rowColumnList[0]["rowIndex"] + 1}
        rowColumnList.append(endRowColumnIndex)

    return rowColumnList


def loadConfiguration(fileName):
    '''This function loads the json configuration'''
    
    configMap = {
    "Endpoint" : "https://cosmos-dfc-edr-prod.documents.azure.com:443/",
    "Masterkey" : "j7WkqqvfU2R0QGAb7mhVFRDFiD2tdfYj32RT17a1Q4iRzNDu0v38oa7XJ8d9osQJ3QQ3VlOV4C2EmePeCLVo5g==",
    "Database" : "dfc-edr",
    "Collection" : "config",
    "preferredRegions" : "West US",
    "query_custom" : "SELECT * FROM c WHERE c.configType = 'preprocess' and RegexMatch ('" + fileName + "', c.fileTypeRegex, 'i')"
    #"query_custom" : "SELECT * FROM c WHERE c.configType = 'preprocess' AND udf.REGEX_MATCH_UDF('" + configFileName + "',c.fileTypeRegex)"
    }

    dataInCosmosDb = spark.read.format("com.microsoft.azure.cosmosdb.spark").options(**configMap).load()
    jsonConfiguration = dataInCosmosDb.select('*').toJSON().collect()
    jsonConfiguration = json.loads(jsonConfiguration[0])
    jsonConfiguration["fileName"] = fileName
    jsonConfiguration["fileNameWithNoExtension"] = fileName.lower().replace((jsonConfiguration["fileTypeFormat"]).lower(), "")
  
    return jsonConfiguration


def loadDatasetFromFile(config):
    '''This function reads the excel file'''

    #remove any leading/trailing "."    
    inputFileTypeFormat = config["fileTypeFormat"].strip(".")    
    
    #filelocation = "/pre_post_processing_landing/"
    filelocation = "/dbfs/mnt/datalakemount/pre-process-landing/"
    fileName = filelocation + config["fileName"]
    df = None
    
    if inputFileTypeFormat == "csv":
      if config["separator"] == ",":
        df = pandas.read_csv(fileName, sep=config["separator"], header=None)
    #if tab delimited then check if encoding is setup, else use chardet to derive encoding   
      if config["separator"] == "\t":
        if 'encoding' in config:
          df = pandas.read_csv(fileName, sep = config["separator"], encoding = config["encoding"], header=None)
        else:
          with open(fileName, 'rb') as rawdata:
            result = chardet.detect(rawdata.read(1000000))
            df = pandas.read_csv(fileName, sep = config["separator"], encoding = result["encoding"], header=None)       
      
      #df = df.replace(to_replace = r'(\\n|\\r)+', value='  ', regex = True)
    else:
      df = pandas.read_excel(fileName, sheet_name=None, header=None)
    
    print("Input file loaded into memory:")
    return df


def cleanseExcel(cleansingConfig, excelDataFrame):
    '''This function replaces any special characters with their equivalents, from the config '''

    excelDataFrame = excelDataFrame.replace(to_replace=cleansingConfig["key"], value=cleansingConfig["value"], regex = True)
    print("Global Cleanisng Complete")
    return excelDataFrame


def cleanseExcelColumn(cleansingConfig, excelDataFrame):
    '''This function replaces any special characters with their equivalents, from the config, foreach column '''


    for column in cleansingConfig:
        excelDataFrame[column] = excelDataFrame[column].replace(to_replace=cleansingConfig[column]["key"], value=cleansingConfig[column]["value"], regex = True)
    print("Column Cleansing Complete")
    return excelDataFrame


def parseDateStringToObject(inputDateString, inputFormat, castDataType):
    '''This function converts a datestring with a given input format to a Python DateTime Object '''

    key =""
    objFormat= ""
    datePartsDictionary = {"dd" : "%d", "mm": "%m", "yy":"%y", "yyyy":"%Y", "hh" : "%H:", "MM" : "%M:", "ss" : "%S:"}
    outputDateTimeObject = None

    for i in range(len(inputFormat)):
        if (inputFormat[i]).isalpha() :
            key += inputFormat[i]
        else:
            objFormat += datePartsDictionary[key] + "/"
            key = ""

    key = datePartsDictionary[key]
    objFormat += key
    outputDateTimeObject = datetime.datetime.strptime(inputDateString.strip(" "), objFormat)

    if castDataType.lower() == "date":
        outputDateTimeObject = outputDateTimeObject.date()

    outputDateFormat = "%Y-%m-%d"
    return str(outputDateTimeObject.strftime(outputDateFormat))


def transformDateTimeObject(transformMap, excelCell):
    '''This function parses and transforms cell content to date/datetime object'''

    #Get the start and end date from the column specified
    transformedData = str(excelCell)

    #Remove additional characters from data as mentioned in config
    transformedData = transformedData.strip("".join(transformMap["removeChars"]))

    if transformMap["token"] != '':
        #Split based on token
        transformedData = transformedData.split(transformMap["token"])

        #Fetch based on tokenizer index
        transformedData = transformedData[transformMap["tokenizerIndex"]]

    #Get current date format in the extracted excel cell
    castFormat = transformMap["castFormat"]
    
    if castFormat != "":
      #Cast the date string with the given input format to a Python DateTime Object
      transformedData = parseDateStringToObject(transformedData, castFormat, transformMap["castDataType"])

    return transformedData


def transformColumns(dataframeConfig, excelDataCell, processedDataset):
    '''This function extracts cell values from the processed excel dataset and injects those cell values as columns'''
    
    #Extracting the cell contents from the dataframe. If datasource for the column needs to be extracted from input filename
    #Then set the source attribute to "fileName"
    if "source" in dataframeConfig and dataframeConfig["source"] == "fileName":
        excelDataCell = dataframeConfig["commonConfig"]["fileName"]
    else:
        excelDataCell = excelDataCell.iloc[0,0]

    #Extract the required configurations
    headersMap = dataframeConfig["headersMap"]
    transformMap = dataframeConfig["transformMap"]
    
    #For each column name in headersmap config, there is a corresponding config in transformMap config.
    #Looping through each of the headers in headersmap
    for transformMapKey in headersMap.values():        

        transformedData = None

        #Fetching the datatype that needs to be processed, for transforming
        castDataType = str(transformMap[transformMapKey]["castDataType"]).lower()

        #If the datatype is Date or DateTime, called the corresponding function to get the transformed data
        if(castDataType in ["date", "datetime"]):
            transformedData = transformDateTimeObject(transformMap[transformMapKey], excelDataCell)

        #If the key "mergeCondition" condition is present in the config, the insert the 
        if "mergeCondition" in dataframeConfig:
            #Getting the merge-index: The position at which this transformed column needs to merged to in the main dataset
            mergeIndex = int(transformMap[transformMapKey]["mergeIndex"])
            processedDataset.insert(mergeIndex, transformMapKey, transformedData)

    return processedDataset


def renameDataFrameColumns(dataframeConfig, excelDataFrame):
    '''Rename current column names to match mapping of headersMap'''

    headersMap = dataframeConfig["headersMap"]

    #Remove header whose key is "?", as this is used for transformation
    if "?" in headersMap : headersMap.pop("?")

    #Rename columns
    excelDataFrame = excelDataFrame.rename(columns = headersMap)
    
    return excelDataFrame

def generateSummaryValidation(datasetConfig, excelDataFrame, minDate, maxDate):

    minDateField = minDate
    maxDateField = maxDate
    if "validations" in datasetConfig and "summaryValidation" in datasetConfig["validations"] and isinstance(datasetConfig["validations"]["summaryValidation"], dict):
        minDateField = datasetConfig["validations"]["summaryValidation"]["minDateField"]
        maxDateField = datasetConfig["validations"]["summaryValidation"]["maxDateField"]

        minDateField = excelDataFrame[minDateField].min()
        maxDateField = excelDataFrame[maxDateField].max()
    
    return { "min": minDateField, "max" : maxDateField }
  

def generateSummary(summaryType, datasetConfig, status, message, excelDataFrame, datasetName = "", dataframeName = ""):
    '''This function generates the required summary based on the summary type passed'''
    
    result = dict()
    minDate = ""
    maxDate = ""
    if datasetName == None : datasetName = ""
    if dataframeName == None : dataframeName = ""
      
    if summaryType.lower() not in ["error", "genericerror"] and {"Start Date","End Date"}.issubset(excelDataFrame.columns.tolist()):
        minDate = excelDataFrame[["Start Date"]].head(1).astype("str")
        maxDate = excelDataFrame[["End Date"]].head(1).astype("str")

    result["status"] = status
    result["DBStatus"] = summaryType
    if summaryType.lower() == "error":
        
        result["error"] = {
                            "databricksLocation": datasetConfig["databricksLocation"],
                            "databricksFile": datasetConfig["databricksFile"],
                            "error": message
                          }
        
    elif summaryType.lower() == "genericerror":
        
        result["genericerror"] = {
                            "Dataset": datasetName,
                            "Dataframe": dataframeName,
                            "error": message
                          }

    elif summaryType.lower() == "warning":
      
        warningCollection = ""        
        if "warning" in result:
            warningCollection = result["warning"]["warning"]
        warningCollection = warningCollection + ("dataset: " + datasetName + ", " + "dataframeName: " + dataframeName + ", message: " + message) + ";"
        
        summaryValidation = generateSummaryValidation(datasetConfig, excelDataFrame, minDate, maxDate)
        result["warning"] = {
                              "databricksLocation": datasetConfig["databricksLocation"],
                              "databricksFile": datasetConfig["databricksFile"],
                              "databricksMasterFile": datasetConfig["databricksMasterFile"],
                              "destinationLocation": datasetConfig["destinationLocation"],
                              "destinationFile": datasetConfig["destinationFile"],
                              "destinationMasterFile": datasetConfig["destinationMasterFile"],
                              "dataset": datasetName,
                              "dataframeName": dataframeName,
                              "warning": str(warningCollection),
                              "summary": {
                                  "fileType": (re.search("(\.\w+)$", datasetConfig["databricksFile"])).group(),
                                  "minDate": summaryValidation["min"],
                                  "maxDate": summaryValidation["max"],
                                  "count": excelDataFrame.shape[0]
                              }
                        }

    else:
      
        summaryValidation = generateSummaryValidation(datasetConfig, excelDataFrame, minDate, maxDate)
        result["success"] =   {
                        "databricksLocation": datasetConfig["databricksLocation"],
                        "databricksFile": datasetConfig["databricksFile"],
                        "databricksMasterFile": datasetConfig["databricksMasterFile"],
                        "destinationLocation": datasetConfig["destinationLocation"],
                        "destinationFile": datasetConfig["destinationFile"],
                        "destinationMasterFile": datasetConfig["destinationMasterFile"],
                        "dataset": datasetName,
                        "dataframeName": dataframeName,
                        "summary": {
                            "fileType": (re.search("(\.\w+)$", datasetConfig["databricksFile"])).group(),
                            "minDate": summaryValidation["min"],
                            "maxDate": summaryValidation["max"],
                            "count": excelDataFrame.shape[0]
                        }
                    }

    return result


def dataframeColumnValidations(validColumnConfig, excelDataFrame):
    '''This validation function checks if the dataframe columns loaded from the excel and in the headersmap object match. Throws exception is not a match'''

    result = None

    #Remove all keys with text "?" as this is the key used for transformation and should not be considered for validation
    if "?" in validColumnConfig : validColumnConfig.pop("?")
    
    
    if len(validColumnConfig) > 0:
        #Get the list of columns mentioned in the config into a 'set' datatype
        validColumnSet = set(validColumnConfig)

        #Get the column header list from the excel
        dataframeColumnSet = set(excelDataFrame.columns.tolist())

        #Check if the excel contains the necessary columns
        if not validColumnSet.issubset(dataframeColumnSet):
            
            #If the excel does not contain all the columns, then identify the missing columns. If true, generate error message.
            validColumnSet.difference_update(dataframeColumnSet)
  
            #Raise execption as no further processing should occur
            raise Exception("Unable to process dataframe. Columns Missing: " + ((",").join(validColumnSet)), "validationFailed")

    print("Key Columns Validated:")
    return result


def dataframeUniqueRowValidation(uniqcolumnList, excelDataFrame):
    '''This function checks for duplicates in the dataframe, by combining the columns passed'''

    uniqueDataCheck = excelDataFrame[uniqcolumnList]
    duplicateRowsDF = uniqueDataCheck.duplicated().unique()
    if True in duplicateRowsDF:
        #Raise execption as no further processing should occur
        raise Exception("Unable to process dataframe. Duplicates present. Duplicate values:", "validationFailed")
    print("Duplicate record check completed:")
    return None


def dataframeAdditionalColumnValidation(dataframe, config, processedDataset):
    '''This function checks for additional columns in the input excel file and raises warning if any is found'''

    result = None
    if (len(dataframe["headersMap"]) > 0) and ("?" not in dataframe["headersMap"]):
        dataframeColumnSet = set(processedDataset.columns.tolist())
        ValidAllColumnSet = set(dataframe["headersMap"].values())
        additionalColumns =  dataframeColumnSet.difference(ValidAllColumnSet)

        if len(additionalColumns) > 0:
          result = "The input dataset has the following additional columns: " + ((",").join(additionalColumns))

    return result


def processDataFrame(dataframeConfig, excelDataset, processedDataset):
    '''This function processes the excel file based on the dataframe config passed'''
    print("processDataFrame")
    worsheetDf = dataframeConfig["worksheet"]
    region = getExcelRegionInIntegers(dataframeConfig["region"])
    headerRow = dataframeConfig["headersRow"]
        
    if dataframeConfig["commonConfig"]["fileTypeFormat"] == "csv":
      #Assign dataframe as worksheet incase of csv
      worsheetDf = excelDataset
      
    else:
      #Extract the particular worksheet from the entire excel dataframe for excel files
      print("#Extract the particular worksheet from the entire excel dataframe for excel files")
      worsheetDf = excelDataset[worsheetDf]
      print(worsheetDf)
      
    #Extract data based on region in config
    worsheetDf = worsheetDf.iloc[region[0]["rowIndex"]:region[1]["rowIndex"], region[0]["colIndex"]:region[1]["colIndex"] + 1]
    
    #When headerRow == -1 ==> The headers are not part of data. They are read from the excel as column headers directly
    #When headerRow > -1 ==> They are part of data, they need to be extracted and assigned to the dataframe as headers.
    if headerRow > -1:
      #Extract the column header names from the worksheet data set, from the given header row location
      excelDataFrameColumns = worsheetDf.iloc[headerRow,]

      #Assign the extracted column names as column headers
      worsheetDf.columns = excelDataFrameColumns.values.tolist()

      #Delete the row from which the column headers were extracted
      worsheetDf = worsheetDf.drop(worsheetDf.index[headerRow])
        
    
    #Cleanse dataframe
    if "cleansingMap" in dataframeConfig and len(dataframeConfig["cleansingMap"]) > 0:
        worsheetDf = cleanseExcelColumn(dataframeConfig["cleansingMap"], worsheetDf)

    #Transform columns
    if "transformMap" in dataframeConfig:
        worsheetDf = transformColumns(dataframeConfig, worsheetDf, processedDataset)

    #Rename column to headersMap dictionary values
    worsheetDf = renameDataFrameColumns(dataframeConfig, worsheetDf)

    return worsheetDf

  
def processFilenamesByConfig(datasetConfig, excelDataset, fileNameWithNoExtension):
    '''This function reads the output files from the config, performs any manipulation that is required on them, reconstructs the new filename and saves it back in config
       As those filenames can be used in anyother function directly e.g. in generateSummary function'''
  
    #static list if files a dataset will have. New files can be added to config and to this list for processing
    filenameList = ["databricksFile", "destinationFile", "destinationMasterFile", "databricksMasterFile"]
    
    #Fetching the default file extension mentioned in the config. By default striping the leading "." as it may or may not be added. So explicitly stripping and adding it
    fileExtension = "." + (datasetConfig["databricksFileExt"]).strip(".")
    
    #Input filename with no extention
    fileNameWithNoExtension = fileNameWithNoExtension.strip(".")

   
    #loop through the list if files
    for fn in filenameList:

        #If that file is present in config, then proceed
        if fn in datasetConfig and len(datasetConfig[fn]) > 0:

            #If "useSourceFileName" is true, then the output file name should be same as input filename and the extension should be as per the format in config
            #Remove filenameReplaceChars config so that there is no processing required
            if (str(datasetConfig["useSourceFileName"]).lower()) == "true":
                finalFileName = fileNameWithNoExtension
                datasetConfig["filenameReplaceChars"] = None
            else:
                finalFileName = datasetConfig[fn]

            #If current key is referring to master file
            if fn.find("Master") > -1:

                #By business rule, the master file will get generated only if Subpropjection property with some columns are present in the config
                #If there is no subprojection, then master file will not get generated.
                if "subProjection" in datasetConfig and len(datasetConfig["subProjection"]) > 0:
                    if (str(datasetConfig["useSourceFileName"]).lower()) == "true":
                        finalFileName = finalFileName + "_master"
                else:
                    finalFileName = ""
         
            #Reconstruct filename based on rules in config property "filenameReplaceChars". 
            #In case "filenameReplaceChars" is empty, then just return filename. FileExtension gets appended if required
            datasetConfig[fn] = reconstructFileNameByConfig(datasetConfig, excelDataset, finalFileName, fileExtension)

    return None

  
def reconstructFileNameByConfig(config, excelDataset, fileName, fileExtension):
    '''This function reconstucts the filename based on config data'''
  
    #if config has info on how to process files names, then re-construct filenames based on those rules. Else just return the filename directly
    if "filenameReplaceChars" in config and config["filenameReplaceChars"] != None:
        #Get the key and value that needs to be replaced in the filename
        fileNameReplaceCharactersKey = next(iter((config["filenameReplaceChars"]).keys()))
        fileNameReplaceCharactersValue = next(iter((config["filenameReplaceChars"]).values()))

        #Extract the value that needs to be replaced in the filename, from the dataframe
        fileNameReplaceCharactersValue = excelDataset[[fileNameReplaceCharactersValue]].head(1).astype("str")
        fileNameReplaceCharactersValue = str((fileNameReplaceCharactersValue.values.tolist())[0]).strip("[]'").replace("-","")

        #Replace the values to the get the final filenames
        fileName = fileName.replace(fileNameReplaceCharactersKey, fileNameReplaceCharactersValue)

    #If the filename does not have an extension, then append the default extension that is present in the config
    #"(\.\w+)$" - This regex checks if the file has an  extension, When no extension, then None is returned.
    if fileName != "" and re.search("(\.\w+)$", fileName) == None:
        fileName = fileName + fileExtension
          
    return fileName

  
def saveDataset(datasetConfig, subProjectedDataset, excelDataset):
    '''This function saves the dataframe in the format mentioned in the config'''    
    #default to Comma Separator
    separator = ","
    if "separator" in datasetConfig:
        separator = datasetConfig["separator"]

    #Save master dataset
    if "databricksMasterFile" in datasetConfig and len(datasetConfig["databricksMasterFile"]) > 0:        
        excelDataset.to_csv(datasetConfig["databricksLocation"] + datasetConfig["databricksMasterFile"], sep=separator, index=False)
        
    #Save sub-projection dataset file
    subProjectedDataset.to_csv(datasetConfig["databricksLocation"] + datasetConfig["databricksFile"], sep=separator, index=False)
    
    return True

def characterWarningCheck(config, excelDataset):
    '''This function checks for special characters in the dataset and resturns a warning if any of them are present'''
    
    message = None
    invalidCharacterList = ""
    #If invalid chatacters are present in config
    if "dataCharacterWarning" in config and len(config["dataCharacterWarning"]) > 0:
        print('characterWarningCheck')
        
        #Loop through each columns in dataset
        for column in excelDataset.columns.tolist():
            #Loop through each invalid chartacter in config
            for invalidCharacter in config["dataCharacterWarning"]:
                #Check if the character exists in given column of the dataset. If yes, then the number of rows returned would be greater than zero
                if len(excelDataset[excelDataset[column].str.contains(invalidCharacter, na=False)]) > 0:
                    invalidCharacterList = invalidCharacterList + "," + invalidCharacter                  

    if len(invalidCharacterList) > 0:
        message = "The input dataset contains the invalid character(s): " + invalidCharacterList.strip(',')
      
    print('characterWarningCheck - Done')
    return message
  
def processDataset(config, excelDataset):
    '''This function processes the excel file based on the array of dataset configs in the config '''
    print("Processing Dataset:")
    processDatasetResults = list()
    subProjectedDataset = None
    warningCollection = None
    print(config["fileName"])
    
    #loop through each dataset object in config
    for dataset in config["datasets"]:
        print("processing dataset:" + dataset["datasetName"])
        processedDataset = None

        try:
          validationResult = None
          processDataFrameResults = dict()
          
          #Loop through each dataframe and process
          for dataframe in dataset["dataframes"]:
              dataframe["commonConfig"] = dict()
              dataframe["commonConfig"]["fileTypeFormat"] = config["fileTypeFormat"]
              dataframe["commonConfig"]["fileName"] = config["fileName"]
              dataframe["commonConfig"]["fileNameWithNoExtension"] = config["fileNameWithNoExtension"]
              processedDataset = processDataFrame(dataframe, excelDataset, processedDataset)

              #Check for additional columns in each dataframe
              validationResult = dataframeAdditionalColumnValidation(dataframe, dataset, processedDataset)

              #If there additional columns, generate the warning, save warning in memory and continue processing
              if validationResult != None:
                  processDataFrameResults[dataframe["worksheet"]] = validationResult
           

          #Checking for invalid characters in the dataset
          #Add the invalid character warning to the list of other warning messages, only once          
          invalidCharacterMessage = characterWarningCheck(config, processedDataset)
          if invalidCharacterMessage != None:
              processDataFrameResults["MainDataSet"] = invalidCharacterMessage
          
          #Global Cleansing
          if "globalCleansingMap" in dataset and len(dataset["globalCleansingMap"]) > 0:
            processedDataset = cleanseExcel(dataset["globalCleansingMap"], processedDataset)
          
          #Update fileNames based on info in config. The new filenames will be saved back in config as they will be used in multiple places.
          #This is done after processing the dataframes as the filename could contain data from the processed dataset.
          processFilenamesByConfig(dataset, processedDataset, config["fileNameWithNoExtension"])
          
          #Perform key column validation. Meaning, only if the min required columns are present proceed with the processing
          dataframeColumnValidations(dataset["validations"]["keyColumns"], processedDataset)

          #Perform unique row validation
          if "uniqueValidation" in dataset["validations"]:
            dataframeUniqueRowValidation(dataset["validations"]["uniqueValidation"], processedDataset)
 
          #Get a subset of columns from the main dataset
          subProjectedDataset = processedDataset
          if "subProjection" in dataset and len(dataset["subProjection"]) > 0:
            subProjectedDataset = pandas.DataFrame(processedDataset[dataset["subProjection"]])
          print("after subProjection")
          #Save the dataset in the file locations mentioned in config
          saveDataset(dataset, subProjectedDataset, processedDataset)
          print("saveDataset")
          
          #If warnings are present then generate the summary for those warnings and return tem. 
          #No success messages when warnings are present. This will return warnings across dataframes
          if len(processDataFrameResults) > 0:
              for dataframeResult in processDataFrameResults:
                  warningCollection = generateSummary("warning", dataset, False, processDataFrameResults[dataframeResult], processedDataset, dataset["datasetName"], dataframeResult)
              processDatasetResults.append(warningCollection)
          else:
              print("processDatasetResults-else")
              #Generate success result as there are no warnings
              processDatasetResults.append(generateSummary("success", dataset, True, None, processedDataset, dataset["datasetName"]))

        except Exception as error:
            errorMessage = ""
            for x in error.args:
              errorMessage += str(x) + ","

            errorMessage = errorMessage.strip(",")
            summary = None
            if "validationFailed" in error.args:
                summary = generateSummary("error", dataset, False, errorMessage, None, dataset["datasetName"])
            else:
                summary = generateSummary("genericerror", None, False, errorMessage, None, dataset["datasetName"])

            processDatasetResults.append(summary)

    return processDatasetResults


def mainMethod(fileName):
    '''This function is the entry point'''
    finalResult = dict()
    try:
        #Load config from cosmoDB
        config = loadConfiguration(fileName)
        
        #Read excel dataset into memory from specified file location
        #Set parameter "header" to "-1" to fetch all sheets
        #config["fileTypeFormat"] = contains the input file format
        excelDataset = loadDatasetFromFile(config)

        #Process the in-memory excel dataset and save it at the respective location
        finalResult = processDataset(config, excelDataset)

    except Exception as genericError:
      #On any error, generate and error message and return the error message
      errorMessage = ""
      for x in genericError.args:
        errorMessage += str(x) + ","           
 
      errorMessage = errorMessage.strip(",")
      finalResult = generateSummary("genericerror", None, False, errorMessage, None)

    finally:
        finalResult = json.dumps(finalResult) 
        return finalResult

      
def mainMethoddebugging(fileName):
    '''This function is the entry point'''
    finalResult = dict()

    #Load config from cosmoDB
    config = loadConfiguration(fileName)
    
    #Read excel dataset into memory from specified file location
    #Set last parameter "header" to "-1" to fetch all sheets
    excelDataset = loadDatasetFromFile(config)

    #Process the in-memory excel dataset and save it at the respective location
    finalResult = processDataset(config, excelDataset)

    finalResult =  json.dumps(finalResult) 
    return finalResult

#Input path of the excel to read should incxlsxe the entire path

#comma seperated
#gFileName = "rank-tracker-desktop-20201103.csv"
#tab delimited
#gFileName = "rank-tracker-desktop-20201106.csv"
#gFileName = "ahrefs_ctr.xlsx"
#gFileName = "amznba_20201010_Sales Diagnostic_Detail View_US.xlsx"
#gFileName = "amznba_20200610_Sales Diagnostic_Detail View_US.xlsx"
#gFileName = "amznba_20201028_Sales Diagnostic_Detail View_US.xlsx"
#gFileName = "amznba_20201103_Subcategory_Net PPM_Detail View_US.xlsx"
#gFileName = "amznba_20200104_Item Comparison and Alternate Purchase Behavior_US.xlsx"
#gFileName = "amznba_20191013_Repeat Purchase Behavior_US.xlsx"
#gFileName = "amznba_20201004_Customer Reviews_US.xlsx"
#gFileName = "amznba_20201103_Geographic Sales Insights_Detail View_US.xlsx"
#gFileName = "amznba_20200711_COGS_Sales Diagnostic_Detail View_US.xlsx"
#gFileName = "DFC POS Report 6.15-21.2020.xlsx"
gFileName = "DFC POS Report 1.01-01.14.2020.xlsx"
#gFileName = "DFC POS Report 5.25-05.31.2020.xlsx"
#gFileName = "Delta Monthly POS Report - 06.23.2020.xlsx"
#gFileName = "DELTA FAUCET COMPANY_Omni_POS_2020_10.xlsx"
#gFileName = "DELTA FAUCET COMPANY_Omni_POS_2020_08.xlsx"
#gFileName = "Delta_Wayfair June14_June20-2020 sales.xlsx"
#gFileName = "Delta_Wayfair June21_June28-2020 sales.xlsx"
#gFileName = "Lowes_Day_Model_FW24_2020.xlsx"
#gFileName = "PreviousMonthProductPerf20201004140001.csv"
#gFileName = "Master Virtual SKU List.xlsx"
#gFileName = "StepAzureBlobExport_20201030050853414.xlsx"
#Newline Char issue
#gFileName = "StepAzureBlobExport_20201030000000000.csv"
#gFileName = "StepAzureBlobExport_20201106081255678.csv"
#tab delimited
#gFileName = "StepAzureBlobExport_20201106081250002.csv"
#gFileName = "StepAzureBlobExport_20201111091211854.csv"
#gFileName = "THD VDF STATUS -DELTA FAUCET COMPANY - 15891_2020_08_11.xlsx"
#gFileName = "unknown_models_09302020.xlsx"
#gFileName = "hd_cal_20201030.xlsx"
#gFileName = "unknown_models_09302020.xlsx"
#gFileName = "unknown_models_20201023.xlsx"
#gFileName = "Wayfair Product Performance Report_202009_20201015_Results.xlsx"
#gFileName = "Wayfair Product Performance Report_20200901_20201015_Results.xlsx"
#mainMethoddebugging(gFileName)
mainMethod(gFileName)


Input file loaded into memory:
Processing Dataset:
DFC POS Report 1.01-01.14.2020.xlsx
processing dataset:POSReturns
processDataFrame
#Extract the particular worksheet from the entire excel dataframe for excel files
 0 1 ... 13 14
0 PONUMBER VENDOR ... SERIES SITE
1 878337549 Ferguson_2920 ... Litze BUILD
2 878337549 Ferguson_2920 ... Litze BUILD
3 878337549 Ferguson_2920 ... Litze BUILD
4 878337549 Ferguson_2920 ... Litze BUILD
5 878337549 Ferguson_2920 ... Sensori BUILD
6 878337549 Ferguson_2920 ... NaN BUILD
7 878337549 Ferguson_2920 ... Litze BUILD
8 878380051 Ferguson_2911 ... Siderna BUILD
9 878549423 Ferguson_039 ... NaN BUILD
10 878748426 Ferguson_474 ... Litze BUILD
11 878748426 Ferguson_474 ... Litze BUILD
12 878944448 Ferguson_2920 ... Litze BUILD
13 879040609 Ferguson_2920 ... RSVP BUILD
14 879040609 Ferguson_2920 ... RSVP BUILD
15 879050442 Ferguson_796 ... Sensori BUILD
16 879050447 Ferguson_039 ... Levoir BUILD
17 879169890 Ferguson_2920 ... Litze BUILD
18 879262396 Ferguson_039 ... Baliza BUILD
19 879367295 Ferguson_761 ... NaN BUILD
20 879453278 Ferguson_533 ... Litze BUILD
21 879483952 Ferguson_321 ... NaN BUILD
22 879497383 Ferguson_2911 ... Litze BUILD
23 879508008 Ferguson_986 ... Artesso BUILD
24 879524744 Ferguson_761 ... NaN BUILD
25 879549516 Ferguson_986 ... Sensori BUILD
26 879549518 Ferguson_796 ... Siderna BUILD
27 879556129 Ferguson_796 ... NaN BUILD
28 879556129 Ferguson_796 ... Odin BUILD
29 879598151 Ferguson_533 ... Levoir BUILD
.. ... ... ... ... ...
523 879889643 Ferguson_474 ... Cassidy BUILD
524 879889643 Ferguson_474 ... MultiChoice BUILD
525 879889643 Ferguson_474 ... MultiChoice BUILD
526 879889643 Ferguson_474 ... NaN BUILD
527 879889643 Ferguson_474 ... NaN BUILD
528 879889643 Ferguson_474 ... Cassidy BUILD
529 879889643 Ferguson_474 ... Cassidy BUILD
530 879889643 Ferguson_474 ... Cassidy BUILD
531 879890189 Ferguson_423 ... Zura BUILD
532 879896550 Ferguson_796 ... Cassidy BUILD
533 879896995 Ferguson_2920 ... Pivotal BUILD
534 879898319 Ferguson_2920 ... Lahara BUILD
535 879899477 Ferguson_986 ... Ara BUILD
536 879903494 Ferguson_321 ... Modern BUILD
537 879906501 Ferguson_986 ... Lahara BUILD
538 879909296 Ferguson_423 ... Trinsic BUILD
539 879910149 Ferguson_321 ... Trinsic BUILD
540 879910462 Ferguson_474 ... Addison BUILD
541 879916053 Ferguson_533 ... Dryden BUILD
542 879916053 Ferguson_533 ... Dryden BUILD
543 879916053 Ferguson_533 ... Dryden BUILD
544 879916053 Ferguson_533 ... Dryden BUILD
545 879917108 Ferguson_2920 ... Vero BUILD
546 879917149 Ferguson_688 ... Trinsic BUILD
547 879918998 Ferguson_688 ... Pivotal BUILD
548 879018639 Ferguson_761 ... Xander BUILD
549 879769524 Ferguson_2920 ... Choice BUILD
550 880053980 Ferguson_986 ... Core BUILD
551 880124368 Ferguson_986 ... NaN BUILD
552 880156597 Ferguson_2920 ... NaN BUILD

[553 rows x 15 columns]
characterWarningCheck - Done
Key Columns Validated:
Duplicate record check completed:
after subProjection
saveDataset
processing dataset:POSPurchases
processDataFrame
#Extract the particular worksheet from the entire excel dataframe for excel files
 0 1 ... 13 14
0 PONUMBER VENDOR ... SERIES SITE
1 880020326 Ferguson_039 ... Odin BUILD
2 880020326 Ferguson_039 ... Odin BUILD
3 880067769 Ferguson_039 ... Traditional BUILD
4 880067769 Ferguson_039 ... NaN BUILD
5 880067769 Ferguson_039 ... HydraChoice BUILD
6 880067769 Ferguson_039 ... HydraChoice BUILD
7 880070550 Ferguson_039 ... Litze BUILD
8 880070550 Ferguson_039 ... NaN BUILD
9 880070550 Ferguson_039 ... Odin BUILD
10 880070550 Ferguson_039 ... Odin BUILD
11 880070550 Ferguson_039 ... Odin BUILD
12 880072408 Ferguson_039 ... NaN BUILD
13 880072408 Ferguson_039 ... Odin BUILD
14 880098148 Ferguson_039 ... Litze BUILD
15 880103857 Ferguson_039 ... Solna BUILD
16 880128431 Ferguson_533 ... Baliza BUILD
17 880128431 Ferguson_533 ... Traditional BUILD
18 880128431 Ferguson_533 ... Traditional BUILD
19 880128431 Ferguson_533 ... Traditional BUILD
20 880128431 Ferguson_533 ..